In [ ]:
import warnings

warnings.filterwarnings("ignore")
from smartbind import load_smartbind_models
from smartbind import logger
from smartbind.preprocess import convert_smiles_to_pf2

import pandas as pd
from torch.nn.functional import cosine_similarity

#### Set up inputs
Please refer to the [README.md](https://github.com/AIDD-LiLab/SMARTBind-internal/blob/main/README.md) to download the SMARTBind pre-trained models. For virtual screening background library, please refer to the following command.
```
gdown --id 16CvakEfkm0qGpHCejcAng_RQvNArfzyG
```

In [ ]:
smiles_txt_path = 'Chemspace_Screening_Compounds_SMILES_sampled1M.smi'
ensembled_models_path = '../../../SMARTBind_weight'
save_path = 'ranking_percentile_scores.txt'
device = 'cpu'
batch_size = 10000

### Load SMARTBind pre-trained model

In [ ]:
logger.info('Get SmartBind pre-trained model objects')
logger.info(f'Loading models from {ensembled_models_path}')
smartbind_models = load_smartbind_models(
    model_path=ensembled_models_path,
    device=device,
    vs_mode=True
)

### Load RNA-small molecule positive pairs

#### Reproduce case study results
In figure 4, there are case studies on 5 RNA-small molecule pairs including 9CPD-MQC, 9CPI-A1AZL, 9CPG-A1AZM, pre-miR-372 (Fragment-3), 5E54 (Apo) / 4TZX (Holo). In order to reproduce the results, simply replace the csv path with `../../data/temporal_test_set/case_study_test_set.csv`.

In [ ]:
temporal_pd = pd.read_csv(f'../../data/temporal_test_set/temporal_test_set.csv')
vs_positive_pair_dict = {}
for idx, row in temporal_pd.iterrows():
    pdb_id = row['PDB ID']
    ligand_id = row['Ligand ID']
    smiles = row['SMILES']
    rna_seq = row['RNA sequence']
    vs_positive_pair_dict[f'{pdb_id}_{ligand_id}'] = {
        'ligand': smiles,
        'rna': rna_seq
    }

### Load background library

In [ ]:
with open(smiles_txt_path, 'r') as f:
    smol_smiles_list = f.readlines()
print(f'Number of small molecules in background library: {len(smol_smiles_list)}')
smol_smiles_list_sampled = [smol_smiles.strip() for smol_smiles in smol_smiles_list]
smol_fp2_list = [convert_smiles_to_pf2(smol_smiles) for smol_smiles in smol_smiles_list_sampled]

### Virtual screening

In [ ]:
for k, v in vs_positive_pair_dict.items():
    print(f'Virtual screening for {k}')
    rna = v['rna']
    ligand = v['ligand']
    smol_fp2 = convert_smiles_to_pf2(ligand)

    rank_result_by_models = {}
    for model in smartbind_models:
        smol_fp2_list_copy = smol_fp2_list.copy()
        rank_result_by_models[smartbind_models.index(model)] = []
        rna_embed = model.inference_single_rna(rna)
        smol_fp2_list_copy.insert(0, smol_fp2)

        # split the list into batches, every batch has 10000 small molecules
        batch_size = 10000
        num_batches = len(smol_fp2_list_copy) // batch_size
        if len(smol_fp2_list_copy) % batch_size != 0:
            num_batches += 1
        for i in range(num_batches):
            start = i * batch_size
            end = (i + 1) * batch_size
            smol_embeds = model.inference_list_smols(smol_fp2_list_copy[start:end])
            similarities = cosine_similarity(rna_embed, smol_embeds).tolist()

            rank_result_by_models[smartbind_models.index(model)].extend(similarities)

    # convert rank_result_by_models to pandas
    df = pd.DataFrame(rank_result_by_models)
    # average by row
    df['average'] = df.mean(axis=1)

    target_average = df['average'][0]
    average_list = df['average'][:].to_list()
    ranked_list = sorted(average_list, reverse=True)
    target_rank = ranked_list.index(target_average) + 1
    print(f'For {k}, the rank is {target_rank} out of {len(average_list)}')
    print(f'Rank percentage: {target_rank / len(average_list) * 100}%')

    with open(save_path, 'a') as log_file:
        log_file.write(f'For {k}, the rank is {target_rank} out of {len(average_list)}\n')
        log_file.write(f'Rank percentage: {target_rank / len(average_list) * 100}%\n\n')